# DocuMind AI — retrieval & refusal evaluation

This notebook reads `eval-results.json`, produced by `node scripts/eval.mjs` against the live deployment.

- **Recall@K** — did a gold page appear in the top K retrieved passages?
- **Precision@K** — what share of the top K passages came from a gold page?
- **MRR** — 1 / rank of the first correct passage, averaged over questions.

Three retrievers are compared on 25 questions with known answer pages: hybrid (vector + keyword, RRF), vector only, keyword only. The refusal set checks that unanswerable questions are refused.

In [ ]:
import json
import pandas as pd

with open('eval-results.json') as f:
    r = json.load(f)
print('Evaluated', r.get('evaluatedAt'), 'on', r.get('target'))
metrics = pd.DataFrame(r['metrics']).T
metrics[['recall@1','recall@3','recall@5','recall@10','precision@5','mrr']].round(3)

In [ ]:
import matplotlib.pyplot as plt

ax = metrics[['recall@1','recall@3','recall@5','recall@10']].T.plot(marker='o', figsize=(7,4))
ax.set_xlabel('K'); ax.set_ylabel('Recall'); ax.set_ylim(0, 1.05); ax.set_title('Recall@K by retriever')
plt.show()

In [ ]:
per = pd.DataFrame(r['perQuestion'])
per['hit@5'] = per.apply(lambda x: any(p in x['gold'] for p in x['hybridTop5']), axis=1)
per[['book','q','gold','hybridTop5','hit@5']]

In [ ]:
ref = r.get('refusal')
if isinstance(ref, dict):
    print({k: v for k, v in ref.items() if k != 'items'})
    display(pd.DataFrame(ref.get('items', [])))